In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import boto3
from io import BytesIO

try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns

try:
    from mlxtend.frequent_patterns import fpgrowth, association_rules
except:
    ! pip install 'mlxtend==0.23.1'
    from mlxtend.frequent_patterns import fpgrowth, association_rules

## Helper Classes

In [ ]:
class AssociationRuleModel:
    def __init__(self, str_uri, str_bucket, str_dirname_output):
        self.str_uri = str_uri
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_basket = None
        self.df_frequent = None
        self.df_rules = None
        self.flt_runtime = None

    def import_data(self):
        self.df_basket = pd.read_parquet(self.str_uri)
        print(f'Basket matrix loaded: {self.df_basket.shape[0]:,} transactions, {self.df_basket.shape[1]:,} items')
        return self

    def find_frequent_itemsets(self, flt_min_support=0.005):
        print(f'\nRunning FP-Growth with min_support={flt_min_support}...')
        flt_start = time.time()
        self.df_frequent = fpgrowth(
            self.df_basket,
            min_support=flt_min_support,
            use_colnames=True,
            max_len=4
        )
        self.flt_runtime = time.time() - flt_start
        self.df_frequent['length'] = self.df_frequent['itemsets'].apply(len)
        print(f'Frequent itemsets found: {len(self.df_frequent):,}')
        print(f'Runtime: {self.flt_runtime:.2f} seconds')
        print(f'\nItemsets by size:')
        print(self.df_frequent['length'].value_counts().sort_index())
        return self

    def generate_rules(self, str_metric='lift', flt_min_threshold=1.0):
        print(f'\nGenerating association rules (metric={str_metric}, min_threshold={flt_min_threshold})...')
        self.df_rules = association_rules(
            self.df_frequent,
            metric=str_metric,
            min_threshold=flt_min_threshold
        )
        # cast metric columns to float (mlxtend 0.23 returns object dtype)
        for str_col in ['support', 'confidence', 'lift', 'leverage', 'conviction']:
            if str_col in self.df_rules.columns:
                self.df_rules[str_col] = pd.to_numeric(self.df_rules[str_col], errors='coerce')
        print(f'Association rules generated: {len(self.df_rules):,}')
        return self

    def plot_support_vs_confidence(self):
        if len(self.df_rules) == 0:
            print('No rules to plot.')
            return self
        fig, ax = plt.subplots(figsize=(10, 7))
        scatter = ax.scatter(
            self.df_rules['support'],
            self.df_rules['confidence'],
            c=self.df_rules['lift'],
            cmap='RdYlBu_r',
            alpha=0.6,
            edgecolors='black',
            linewidth=0.5
        )
        plt.colorbar(scatter, label='Lift')
        ax.set_title('Association Rules: Support vs Confidence (colored by Lift)', fontsize=14, y=1.02)
        ax.set_xlabel('Support')
        ax.set_ylabel('Confidence')
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_support_vs_confidence.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_top_rules_by_lift(self, int_top_n=20):
        if len(self.df_rules) == 0:
            print('No rules to plot.')
            return self
        int_top_n = min(int_top_n, len(self.df_rules))
        df_top = self.df_rules.nlargest(int_top_n, 'lift').copy()
        df_top['rule'] = df_top.apply(
            lambda r: '{' + ', '.join(r['antecedents']) + '} → {' + ', '.join(r['consequents']) + '}',
            axis=1
        )
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.barh(range(len(df_top)), df_top['lift'].values, color='#4C72B0', edgecolor='black')
        ax.set_yticks(range(len(df_top)))
        ax.set_yticklabels(df_top['rule'].values, fontsize=9)
        ax.set_title(f'Top {int_top_n} Association Rules by Lift', fontsize=14, y=1.02)
        ax.set_xlabel('Lift')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_top_rules_by_lift.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_metric_distributions(self):
        if len(self.df_rules) == 0:
            print('No rules to plot.')
            return self
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for idx, str_col in enumerate(['support', 'confidence', 'lift']):
            axes[idx].hist(self.df_rules[str_col], bins=40, color='#4C72B0', edgecolor='black', alpha=0.8)
            axes[idx].set_title(f'{str_col.capitalize()} Distribution', fontsize=14, y=1.02)
            axes[idx].set_xlabel(str_col.capitalize())
            axes[idx].set_ylabel('Number of Rules')
            axes[idx].axvline(self.df_rules[str_col].median(), color='red', linestyle='--',
                              label=f'Median: {self.df_rules[str_col].median():.3f}')
            axes[idx].legend()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_metric_distributions.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def save_results_to_s3(self):
        df_save = self.df_rules.copy()
        df_save['antecedents'] = df_save['antecedents'].apply(lambda x: ', '.join(list(x)))
        df_save['consequents'] = df_save['consequents'].apply(lambda x: ', '.join(list(x)))
        str_s3_key = '03_association_rules/association_rules.parquet'
        print(f'\nSaving rules to S3: s3://{self.str_bucket}/{str_s3_key}')
        buffer = BytesIO()
        df_save.to_parquet(buffer, index=False)
        buffer.seek(0)
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key=str_s3_key,
                Body=buffer.getvalue()
            )
            print('Successfully uploaded rules to S3')
        except Exception as e:
            print(f'Error uploading to S3: {e}')
        return self

    def print_top_rules(self, int_top_n=10):
        if len(self.df_rules) == 0:
            print('No rules generated.')
            return self
        int_top_n = min(int_top_n, len(self.df_rules))
        print(f'\n=== TOP {int_top_n} RULES BY LIFT ===')
        df_top = self.df_rules.nlargest(int_top_n, 'lift')
        for _, row in df_top.iterrows():
            str_ante = ', '.join(list(row['antecedents']))
            str_cons = ', '.join(list(row['consequents']))
            print(f'  {{{str_ante}}} → {{{str_cons}}}  '
                  f'(support={row["support"]:.4f}, confidence={row["confidence"]:.4f}, lift={row["lift"]:.4f})')
        return self

    def print_summary(self):
        print('\n=== ASSOCIATION RULES SUMMARY ===')
        print(f'Frequent itemsets: {len(self.df_frequent):,}')
        print(f'Association rules: {len(self.df_rules):,}')
        print(f'Runtime: {self.flt_runtime:.2f} seconds')
        if len(self.df_rules) > 0:
            print(f'\nRule metrics:')
            for str_col in ['support', 'confidence', 'lift', 'leverage', 'conviction']:
                if str_col in self.df_rules.columns:
                    print(f'  {str_col}: mean={self.df_rules[str_col].mean():.4f}, '
                          f'median={self.df_rules[str_col].median():.4f}, '
                          f'max={self.df_rules[str_col].max():.4f}')
        return self

## Constants

In [ ]:
str_bucket = 'market-basket-analysis-demo'
str_task = 'association_rules'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/02_preprocessing/basket_matrix.parquet'

## Output Directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

Output directory ready: ./output


## Run Association Rules

In [ ]:
cls_model = AssociationRuleModel(str_uri, str_bucket, str_dirname_output)
cls_model.import_data()

In [ ]:
cls_model.find_frequent_itemsets(flt_min_support=0.005)

In [ ]:
cls_model.generate_rules(str_metric='lift', flt_min_threshold=1.0)

In [ ]:
cls_model.plot_support_vs_confidence()

In [ ]:
cls_model.plot_top_rules_by_lift()

In [ ]:
cls_model.plot_metric_distributions()

In [ ]:
cls_model.print_top_rules()

In [ ]:
cls_model.print_summary()

In [ ]:
cls_model.save_results_to_s3()

## Completion

In [ ]:
print('\n=== ASSOCIATION RULES COMPLETE ===')
print(f'Rules saved to: s3://{str_bucket}/03_association_rules/association_rules.parquet')